In [ ]:
import Pkg 
Pkg.activate("./..")

In [ ]:
import QuantumGrav as QG
import Flux 
import CUDA 
import GraphNeuralNetworks as GNN
import CausalSets as CS
import Arrow
import Tables

We use `GraphNeuralNetworks.jl` for building the variational autoencoder. This package has many kinds of graph-specific layers already implemented, see the [docs fo more](https://juliagraphs.org/GraphNeuralNetworks.jl/docs/GraphNeuralNetworks.jl/stable/api/basic/). In order to use it, we first need to find out: 
- which data to use 
- how to turn the tabular data into the datatype the GNN package needs
- how to use the GNN package with batches 

In [ ]:
path_to_data = joinpath("/", "media","hmack","dataLinux","machinelearning_data","QuantumGrav","unequal_size_data_small")

In [ ]:
list_of_files = filter(x -> occursin(".arrow", x),readdir(path_to_data));
list_of_files[1:5]

### First idea for which data to use 
The primary information that we use is in the graph structure, not the nodes individually. However, in order to augment the system with more information, we can try and use the coordinates and the past and future relations on each node and treat them as node features. This way, we pass the network information about the causal sets: 
- the manifold they are embedded in 
- its dimensionality 
- the structure of the graph from node to node

We need to first find a good way to encode this information and pass it to the network. 

In [ ]:
first_batch = Arrow.Table(joinpath(path_to_data, list_of_files[1]));

features to use for now: coords, linkMatrix, past relations, future relations, coords

In [ ]:
past_relations = first_batch.past_relations;
future_relations = first_batch.future_relations;
coords = first_batch.coords;
linkMatrix = first_batch.linkMatrix;
atomCount = Int.(first_batch.n);
nmax = Int.(first_batch.nmax[1]);

In [ ]:
to_matrix(x) = hcat(x...); 

In [ ]:
function collate_matrix(x::Matrix{T}, n::Int64, m::Int64)::Matrix{T} where T 
    mat = zeros(T, n, m);

    n_, m_ = size(x);
    mat[1:n_, 1:m_] = x;
    return mat;
end

In [ ]:
function encode_data(n_max, linkMatrix, past_relations, future_relations, coords)::GNN.GNNGraph

    n_nodes = size(past_relations, 1)

    linkMatrix = collate_matrix(reshape(linkMatrix, (n_nodes, n_nodes)), n_max, n_max) 

    past_relations = collate_matrix(reshape(past_relations, n_nodes,:), n_max, n_max)

    future_relations = collate_matrix(reshape(future_relations, n_nodes,:), n_max, n_max)

    coords = collate_matrix(reshape(coords, :, n_nodes), 2, n_max)
    
    return GNN.GNNGraph(linkMatrix, ndata = (past_relations=past_relations, future_relations=future_relations, coords=coords))
end

pay attention to the vectorization syntax in Julia func.(v) vectorizes func over each elements of v. this is easily missed.

In [ ]:
encoded = encode_data.(nmax, to_matrix.(linkMatrix), to_matrix.(past_relations), to_matrix.(future_relations), to_matrix.(coords));

In [ ]:
first(encoded)

In [ ]:
GNN.batch(encoded)

### The simplest way: Graph convolutional network

We do this by hand for now, and later we can use `AutoEncoderToolkit.jl`. Graph-convolutional networks only use the adjacency-, i.e., link matrix of the graph, and hence are only structural in nature.

**Remark:** Message-passing networks (underlying all GNNs and actually also other stuff like CNNs) have the same discriminative power as the Weisfeiler-Lehman Graph Isomorphism Test. Hence some non-isomorphic graphs cannot be distinguished by them. Will this be a problem? 

In [ ]:
mutable struct VAE{E, D}
    encoder::E
    decoder::D
end

Flux.@layer VAE

function (vae::VAE)(x::AbstractArray)
    # Encode input to get latent distribution parameters
    (µ, logσ²) = vae.encoder(x)
    
    # Sample z from the latent distribution
    z = reparameterize(µ, logσ²)
    
    # Decode z to get reconstruction
    x̂ = vae.decoder(z)
    
    return x̂, µ, logσ²
end

function reparameterize(µ, logσ²)
    # Sample from the standard normal distribution
    ε = randn(size(µ))
    
    # Reparameterization trick
    z = µ .+ exp.(0.5 .* logσ²) .* ε
    
    return z
end

We use a simpler approach for the data first - only use the adjacency matrix

In [ ]:
function encode_data(n_max, n_nodes, linkMatrix)::GNN.GNNGraph

    linkMatrix = collate_matrix(reshape(collect(linkMatrix), (n_nodes, n_nodes)), n_max, n_max)
    
    return GNN.GNNGraph(linkMatrix,)
end

use a subset of the data to avoid running out of memory

In [ ]:
using StatsBase 

In [ ]:
dset = QG.DataLoader.Dataset(
    path_to_data, 
    sample(list_of_files, Int(length(list_of_files)/10)),
    cache_size = 5,
    columns = [:past_relations, :future_relations, :coords, :linkMatrix, :n, :nmax],
)

the Flux/MLUtils dataloader is almost useless here because it attempts to load all data at once. What's the point of this thing?? Just use it for now because we are operating on small datasets. Maybe use `DTables.jl` later?
Better version is to generate the data randomly from the start

In [ ]:
shuffle_loader = Flux.DataLoader(
    dset,
    batchsize = 128,
    shuffle = true,
)

hence we operate only on the dataset, which does not do this.

we know `nmax` from the generation process in this case: `1.1*10^dimension` 

In [ ]:
nmax = convert(Int64, round(1.1*10^2))

graph batches then can be made like this: 

In [ ]:
i = 1
for batch in shuffle_loader
    batch = GNN.batch([encode_data(nmax, Int(element.n), element.linkMatrix) for element in batch ])

    println("Batch number: ", i, ", ", batch)

    # train model here...

    i += 1

    if i > 3  # just for demo, break after 3 batches
        break
    end
end